In [2]:
from barista_bench.eval import run_model
from barista_bench.schema import Order, OrderList
from barista_bench.prompt.default import BaristaPrompt
from barista_bench.eval.run_model import load_training_set

from barista_bench.eval.run_model import evaluate_api_model
%load_ext autoreload
%autoreload 2



df = load_training_set('../data/train.csv')
df['expected_order'] = df['expected_json'].apply(
    lambda x: OrderList.model_validate_json(x)
)
sample = df[df['expected_json'].apply(len) > 300].sample(3)
examples = sample[['order', 'expected_json']].values.tolist()
df = df[~df.index.isin(sample.index)]
prompt = BaristaPrompt('default.txt', examples)

out = evaluate_api_model('qwen/qwen-2.5-7b-instruct',prompt, df.sample(10))
out

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


100%|██████████| 10/10 [00:25<00:00,  2.54s/it]


,id,order,expected_json,expected_order,pred,loss
317,317,Start me off with four Short COLD BREW include...,"{""items"": [{""name"": ""Cold Brew"", ""size"": ""Shor...","items=[Order(name='Cold Brew', size='Short', q...","{""items"":[{""name"":""Cold Brew"",""size"":""Short"",""...",13.50
44,44,"I'll have a short Cold Brew, paired with a Tal...","{""items"": [{""name"": ""Cold Brew"", ""size"": ""Shor...","items=[Order(name='Cold Brew', size='Short', q...","{""items"":[{""name"":""Cold Brew"",""size"":""Short"",""...",5.80
103,103,"I'll take a Short iced coffee, like wait, chan...","{""items"": [{""name"": ""Iced Coffee"", ""size"": ""Sh...","items=[Order(name='Iced Coffee', size='Short',...","{""items"":[{""name"":""Iced Coffee"",""size"":""Short""...",12.50
384,384,Hook me up with couple of venti hot chocolates...,"{""items"": [{""name"": ""Hot Chocolate"", ""size"": ""...","items=[Order(name='Hot Chocolate', size='Venti...","{""items"":[{""name"":""Hot Chocolate"",""size"":""Vent...",16.80
483,483,Let me get pair of Grande EARL GREY TEA and cl...,"{""items"": [{""name"": ""Earl Grey Tea"", ""size"": ""...","items=[Order(name='Earl Grey Tea', size='Grand...","{""items"":[{""name"":""Earl Grey Tea"",""size"":""Gran...",6.60
400,400,Can I get five grande frappe (coffee) uh add C...,"{""items"": [{""name"": ""Frappe (Coffee)"", ""size"":...","items=[Order(name='Frappe (Coffee)', size='Gra...","{""items"":[{""name"":""Frappe (Coffee)"",""size"":""Gr...",29.40
99,99,Hook me up with one venti caramel macchiato. N...,"{""items"": [{""name"": ""Caramel Macchiato"", ""size...","items=[Order(name='Caramel Macchiato', size='V...","{""items"":[{""name"":""Caramel Macchiato"",""size"":""...",7.75
318,318,"May I get triple you know, short flat whites, ...","{""items"": [{""name"": ""Flat White"", ""size"": ""Sho...","items=[Order(name='Flat White', size='Short', ...","{""items"":[{""name"":""Flat White"",""size"":""Short"",...",10.00
411,411,Grab me pair of like tall Cold Brew.,"{""items"": [{""name"": ""Cold Brew"", ""size"": ""Tall...","items=[Order(name='Cold Brew', size='Tall', qu...","{""items"":[{""name"":""Cold Brew"",""size"":""Tall"",""q...",0.00
214,214,"Start me off with a few bagels wait no, make i...","{""items"": [{""name"": ""Bagel"", ""size"": null, ""qu...","items=[Order(name='Bagel', size=None, quantity...","{""items"":[{""name"":""Bagel"",""size"":null,""quantit...",10.55


In [7]:
print(out[out['loss'] > 20]['expected_json'].iloc[0])

{"items": [{"name": "Frappe (Coffee)", "size": "Grande", "quantity": 5, "modifiers": ["Coconut Milk"]}, {"name": "Strawberry Smoothie", "size": "Tall", "quantity": 2, "modifiers": ["Cold Foam", "Caramel Drizzle"]}, {"name": "Mocha", "size": "Short", "quantity": 2, "modifiers": ["Peppermint Syrup"]}], "total_price": 59.0}


In [8]:
print(out[out['loss'] > 20]['pred'].iloc[0])

{"items":[{"name":"Frappe (Coffee)","size":"Grande","quantity":5,"modifiers":["Coconut Milk"]},{"name":"Strawberry Smoothie","size":"Tall","quantity":2,"modifiers":["Cold Foam","Caramel Drizzle"]},{"name":"Mocha","size":"Short","quantity":2,"modifiers":["Peppermint Syrup"]}],"total_price":29.6}


In [32]:
import re 

ptn =  r'```json\s*([\s\S]*?)\s*```'

json_code = out.iloc[1]['pred']['choices'][0]['message']['content']
matches = re.findall(ptn, json_code)
OrderList.model_validate_json(matches[0])
# matches
# matches[0]

OrderList(items=[Order(name='Cold Brew', size='Short', quantity=1, modifiers=[]), Order(name='Iced Coffee', size='Tall', quantity=1, modifiers=['Hazelnut Syrup'])], total_price=9.75)

In [30]:
o1 = {
    'name' : 'Espresso',
    'size': 'Short',
    'quantity':2 ,
    'modifiers': ['whip cream']

}
o2 = {
    'name' : 'Espresso',
    'size': 'Short',
    'quantity':2 ,
    'modifiers': ['extra shot']

}
o1 = Order.model_validate(o1)
o2 = Order.model_validate(o2)

run_model.order_loss(o1, o2)

0.5

,id,order,expected_json,expected_order
0,0,Lemme get one tall Strawberry Smoothie include...,"{""items"": [{""name"": ""Drip Coffee"", ""size"": ""Ve...","items=[Order(name='Drip Coffee', size='Venti',..."
1,1,I'd like to order single trenta espresso add o...,"{""items"": [{""name"": ""Espresso"", ""size"": ""Trent...","items=[Order(name='Espresso', size='Trenta', q..."
2,2,Could I have single trenta mocha plus peppermi...,"{""items"": [{""name"": ""Mocha"", ""size"": ""Trenta"",...","items=[Order(name='Mocha', size='Trenta', quan..."
3,3,Grab me four avocado toasts.,"{""items"": [{""name"": ""Avocado Toast"", ""size"": n...","items=[Order(name='Avocado Toast', size=None, ..."
4,4,I'm craving couple of VENTI frappe (mocha)s in...,"{""items"": [{""name"": ""Frappe (Mocha)"", ""size"": ...","items=[Order(name='Frappe (Mocha)', size='Vent..."
...,...,...,...,...
495,495,May I get two Tall iced coffees include skim m...,"{""items"": [{""name"": ""Iced Coffee"", ""size"": ""Ta...","items=[Order(name='Iced Coffee', size='Tall', ..."
496,496,I'm feeling like four trenta flat white add oa...,"{""items"": [{""name"": ""Flat White"", ""size"": ""Tre...","items=[Order(name='Flat White', size='Trenta',..."
497,497,Start me off with couple of tall Flat Whites a...,"{""items"": [{""name"": ""Flat White"", ""size"": ""Tal...","items=[Order(name='Flat White', size='Tall', q..."
498,498,Grab me two short like Cold Brews... wait canc...,"{""items"": [{""name"": ""Strawberry Smoothie"", ""si...","items=[Order(name='Strawberry Smoothie', size=..."


In [ ]:
sample = df[df['expected_json'].apply(len) > 300].sample(3)
# not_sample = sample.index
examples = sample[['order', 'expected_json']].values.tolist()
# df.loc[not_sample]
df = df[~df.index.isin(sample.index)]




out = prompt.format_llm_input('I would like a large latte with almond milk and a blueberry muffin.')
out

[{'role': 'system',
  'content': 'You are a Barista POS System designed to take customer orders and output JSON corresponding to what the customer has ordered. The cafe that you are working with has the following menu and instructions:\n\n# BARISTA BENCH: OFFICIAL MENU & RULES (V2)\n\n## 1. HOT COFFEES\n- Espresso: $3.00 (Default: Solo)\n- Americano: $3.50\n- Drip Coffee: $2.50\n- Latte: $4.50 (Default: Whole Milk)\n- Cappuccino: $4.50 (Default: Whole Milk)\n- Flat White: $4.75 (Default: Whole Milk)\n- Mocha: $5.00 (Default: Whole Milk, Whip)\n- Caramel Macchiato: $5.25 (Default: Whole Milk, Drizzle)\n\n## 2. COLD / BLENDED\n- Cold Brew: $4.25\n- Iced Coffee: $3.00\n- Frappe (Coffee): $5.50 (Default: Whole Milk, Whip)\n- Frappe (Mocha): $5.75 (Default: Whole Milk, Whip)\n- Strawberry Smoothie: $6.00 (No Milk)\n\n## 3. TEAS & OTHERS\n- Chai Latte: $4.75 (Default: Whole Milk)\n- Matcha Latte: $5.25 (Default: Whole Milk)\n- Earl Grey Tea: $3.00\n- Green Tea: $3.00\n- Hot Chocolate: $4.00 

In [ ]:
from barista_bench.eval.run_model import evaluate_api_model

out = evaluate_api_model('qwen/qwen-2.5-7b-instruct',prompt, df)
out

 36%|███▌      | 168/464 [07:44<14:22,  2.91s/it]

In [64]:
out['choices'][0]['message']['content']

'{\n    "items": [\n        {"name": "Strawberry Smoothie", "size": "Tall", "quantity": 1, "modifiers": []},\n        {"name": "Drip Coffee", "size": "Venti", "quantity": 1, "modifiers": ["Extra Hot"]},\n        {"name": "Chai Latte", "size": "Trenta", "quantity": 3, "modifiers": ["Caramel Drizzle"]},\n        {"name": "Mocha", "size": "Short", "quantity": 2, "modifiers": ["Sugar Free Vanilla"]}\n    ],\n    "total_price": 23.05\n}'

In [18]:

for ol in df['expected_order']:
    # print(type(ol))
    for it in ol.items:
        if not len(it.modifiers):
            print(it)
    # break

name='Mocha' size='Short' quantity=2 modifiers=[]
name='Earl Grey Tea' size='Tall' quantity=1 modifiers=[]
name='Avocado Toast' size=None quantity=4 modifiers=[]
name='Iced Coffee' size='Tall' quantity=1 modifiers=[]
name='Butter Croissant' size=None quantity=4 modifiers=[]
name='Green Tea' size='Trenta' quantity=1 modifiers=[]
name='Chai Latte' size='Tall' quantity=2 modifiers=[]
name='Matcha Latte' size='Short' quantity=1 modifiers=[]
name='Earl Grey Tea' size='Venti' quantity=3 modifiers=[]
name='Bagel' size=None quantity=2 modifiers=[]
name='Bacon Gouda Sandwich' size=None quantity=3 modifiers=[]
name='Avocado Toast' size=None quantity=2 modifiers=[]
name='Frappe (Mocha)' size='Venti' quantity=3 modifiers=[]
name='Mocha' size='Tall' quantity=1 modifiers=[]
name='Matcha Latte' size='Short' quantity=3 modifiers=[]
name='Caramel Macchiato' size='Trenta' quantity=3 modifiers=[]
name='Earl Grey Tea' size='Venti' quantity=1 modifiers=[]
name='Green Tea' size='Trenta' quantity=3 modifiers

In [1]:
from barista_bench.prompt.default import BaristaPrompt

%load_ext autoreload
%autoreload 2

In [4]:
prompt = BaristaPrompt('default.txt', ())
print(prompt)

You are a Barista POS System designed to take customer orders and output JSON corresponding to what the customer has ordered. The cafe that you are working with has the following menu and instructions:

# BARISTA BENCH: OFFICIAL MENU & RULES (V2)

## 1. HOT COFFEES
- Espresso: $3.00 (Default: Solo)
- Americano: $3.50
- Drip Coffee: $2.50
- Latte: $4.50 (Default: Whole Milk)
- Cappuccino: $4.50 (Default: Whole Milk)
- Flat White: $4.75 (Default: Whole Milk)
- Mocha: $5.00 (Default: Whole Milk, Whip)
- Caramel Macchiato: $5.25 (Default: Whole Milk, Drizzle)

## 2. COLD / BLENDED
- Cold Brew: $4.25
- Iced Coffee: $3.00
- Frappe (Coffee): $5.50 (Default: Whole Milk, Whip)
- Frappe (Mocha): $5.75 (Default: Whole Milk, Whip)
- Strawberry Smoothie: $6.00 (No Milk)

## 3. TEAS & OTHERS
- Chai Latte: $4.75 (Default: Whole Milk)
- Matcha Latte: $5.25 (Default: Whole Milk)
- Earl Grey Tea: $3.00
- Green Tea: $3.00
- Hot Chocolate: $4.00 (Default: Whole Milk, Whip)

## 4. FOOD (No modifiers allowe